<a href="https://colab.research.google.com/github/CokieLee/ProbML-Connections-Project/blob/ananya/green.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install gensim pulp numpy

import numpy as np
from gensim import downloader as api
from itertools import combinations
from collections import defaultdict
from pulp import LpProblem, LpMaximize, LpVariable, lpSum, LpBinary, PULP_CBC_CMD

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.2 MB/s eta 0:00:00


In [ ]:
!pip -q install nltk wordfreq

import re
from collections import defaultdict
import random

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn

from wordfreq import zipf_frequency

def is_good_word(w: str):
    # single token, letters only, not too short/long, no underscores/hyphens
    if "_" in w or "-" in w or " " in w:
        return False

    if not re.fullmatch(r"[A-Za-z]+", w):
        return False

    if len(w) < 3 or len(w) > 12:
        return False

    # common-ish in English (tune threshold)
    if zipf_frequency(w.lower(), "en") < 4:
        return False
    return True

def pick_4_unique(words):
    words = list(dict.fromkeys(words))  # preserve order, unique
    if len(words) < 4:
        return None
    return tuple(words[:4])

#Synonym categories from WordNet synsets
def generate_synonym_categories(max_categories=3000):
    cats = []
    for syn in wn.all_synsets():
        lemmas = [l.name() for l in syn.lemmas()]
        lemmas = [w for w in lemmas if is_good_word(w)]
        # remove trivial variants (plural etc.) is optional; keep it simple for now
        lemmas = sorted(set(lemmas), key=lambda w: -zipf_frequency(w.lower(), "en"))
        if len(lemmas) >= 3.5:
            group = pick_4_unique(lemmas)
            if group:
                cats.append({
                    "type": "synonyms",
                    "label": syn.definition()[:60],  # not perfect, but useful
                    "words": group,
                    "synset": syn.name()
                })
        if len(cats) >= max_categories:
            break
    return cats

#"Types of X" categories: pick a hypernym, sample 4 hyponyms (nouns only)
def generate_typeof_categories(max_categories=3000, min_hyponyms=8):
    cats = []
    noun_synsets = list(wn.all_synsets(pos=wn.NOUN))
    for parent in noun_synsets:
        # collect hyponyms (direct + maybe 2 hops)
        hypos = set(parent.hyponyms())
        for h in list(hypos):
            hypos.update(h.hyponyms())

        # turn hyponyms into candidate words using their lemmas
        words = []
        for h in hypos:
            for l in h.lemmas():
                w = l.name()
                if is_good_word(w):
                    words.append(w)

        # rank by frequency, keep unique
        words = sorted(set(words), key=lambda w: -zipf_frequency(w.lower(), "en"))
        if len(words) >= min_hyponyms:
            group = pick_4_unique(words)
            if group:
                parent_names = [l.name() for l in parent.lemmas()]
                parent_names = [w for w in parent_names if is_good_word(w)]
                parent_names = sorted(set(parent_names), key=lambda w: -zipf_frequency(w.lower(),"en"))
                label = parent_names[0].lower() if parent_names else parent.definition().split()[0].lower()

                cats.append({
                    "type": "types_of",
                    "label": f"types of {label}",
                    "words": group,
                    "parent_synset": parent.name()
                })
        if len(cats) >= max_categories:
            break

    return cats

#Green tends to have things like "all color ____"
COLOR_WORDS = sorted({
    "red","blue","green","yellow","orange","purple","violet","indigo","pink","brown","black","white","gray","grey",
    "cyan","teal","navy","maroon","magenta","lavender","beige","tan","cream","ivory","turquoise","aqua",
    "scarlet","crimson","ruby","emerald","sapphire","amber","gold","silver","bronze","peach","coral","mint","olive"
})

def generate_color_categories():
    usable = [c for c in COLOR_WORDS if is_good_word(c)]
    cats = []
    for _ in range(200):
        group = tuple(random.sample(usable, 4))
        cats.append({"type": "colors", "label": "colors", "words": group})
    return cats

#make the bank
random.seed(7)
syn_cats = generate_synonym_categories(max_categories=1500)
type_cats = generate_typeof_categories(max_categories=1500, min_hyponyms=10)
color_cats = generate_color_categories()
category_bank = syn_cats + type_cats + color_cats

print("Category bank size:", len(category_bank))
print("Examples:")
for c in random.sample(category_bank, 5):
    print(c["type"], "|", c["label"], "|", c["words"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 3.7 MB/s eta 0:00:00
Category bank size: 950
Examples:
synonyms | discover or determine the existence, presence, or fact of | ('find', 'notice', 'discover', 'observe')
synonyms | very drunk | ('tight', 'wet', 'pissed', 'loaded')
colors | colors | ('grey', 'gray', 'navy', 'green')
types_of | types of play | ('check', 'cut', 'shot', 'break')
colors | colors | ('yellow', 'pink', 'blue', 'green')


In [ ]:
def polysemy_count(w):
    return len(wn.synsets(w.lower()))

def generate_puzzle(category_bank, max_tries=5000):
    bank = category_bank[:]
    for _ in range(max_tries):
        chosen = random.sample(bank, 4)
        words = [w for cat in chosen for w in cat["words"]]

        # basic uniqueness
        if len(set(words)) != 16:
            continue

        # avoid highly ambiguous words (tune threshold)
        if any(polysemy_count(w) > 8 for w in words):
            continue

        # simple collision check: do any 2 categories share "too related" words?
        # (cheap version: avoid overlap in lemma forms already handled by uniqueness)
        return chosen, words


    return None, None

chosen_cats, puzzle_words = generate_puzzle(category_bank)
print("Puzzle words:", puzzle_words)
print("\nCategories:")

Puzzle words: None

Categories:


In [ ]:
!pip -q install nltk wordfreq
import re, random
from collections import Counter, defaultdict

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency

# ---------------------------
# helpers
# ---------------------------
def is_good_word(w: str):
    if "_" in w or "-" in w or " " in w:
        return False
    if not re.fullmatch(r"[A-Za-z]+", w):
        return False
    if len(w) < 3 or len(w) > 12:
        return False
    # keep reasonably common words (tune if needed)
    if zipf_frequency(w.lower(), "en") < 3.5:
        return False
    return True

def polysemy_count(w):
    return len(wn.synsets(w.lower()))

def normalize_word(w):
    return w.strip()


def build_word_to_catfreq(category_bank):
    """Counts how many categories each word appears in."""
    freq = Counter()
    for cat in category_bank:
        for w in cat["words"]:
            freq[w.lower()] += 1
    return freq

def clean_category_bank(category_bank,
                        max_word_collision=2,
                        max_word_polysemy=10,
                        max_cat_avg_polysemy=8.0):
    """
    - Drop categories containing words that appear in too many categories.
    - Drop categories containing very polysemous words.
    - Drop categories whose average polysemy is too high.
    """
    # First pass collision counts
    word_catfreq = build_word_to_catfreq(category_bank)

    cleaned = []
    for cat in category_bank:
        ws = [normalize_word(w) for w in cat["words"]]
        ws_low = [w.lower() for w in ws]

        # keep only good words (in case any slipped in)
        if any(not is_good_word(w) for w in ws):
            continue

        # collision filter
        if any(word_catfreq[w] > max_word_collision for w in ws_low):
            continue

        # polysemy filter
        polys = [polysemy_count(w) for w in ws]
        if any(p > max_word_polysemy for p in polys):
            continue
        if (sum(polys)/len(polys)) > max_cat_avg_polysemy:
            continue

        cleaned.append(cat)

    # recompute collisions after removal (optional second pass helps)
    word_catfreq2 = build_word_to_catfreq(cleaned)
    cleaned2 = []
    for cat in cleaned:
        ws_low = [w.lower() for w in cat["words"]]
        if any(word_catfreq2[w] > max_word_collision for w in ws_low):
            continue
        cleaned2.append(cat)

    return cleaned2

print("Original bank:", len(category_bank))
category_bank_clean = clean_category_bank(category_bank,
                                         max_word_collision=2,
                                         max_word_polysemy=10,
                                         max_cat_avg_polysemy=8.0)
print("Clean bank:", len(category_bank_clean))

Original bank: 950
Clean bank: 28


In [ ]:
def generate_puzzle_from_bank(bank, max_tries=30000, seed=7):
    random.seed(seed)
    for _ in range(max_tries):
        chosen = random.sample(bank, 4)
        words = [w for c in chosen for w in c["words"]]
        if len(set(w.lower() for w in words)) != 16:
            continue
        return chosen, words
    return None, None

chosen_cats, puzzle_words = generate_puzzle_from_bank(category_bank_clean, max_tries=50000, seed=7)

if chosen_cats is None:
    print("No puzzle found. Try relaxing cleaning thresholds or increase bank size.")
else:
    print("Puzzle words:", puzzle_words)
    print("\nCategories:")
    for c in chosen_cats:
        print("-", c["type"], "|", c["label"], "|", c["words"])

Puzzle words: ['now', 'immediately', 'directly', 'instantly', 'but', 'just', 'only', 'simply', 'fuck', 'fucking', 'ass', 'screw', 'respect', 'honor', 'honour', 'observe']

Categories:
- synonyms | without delay or hesitation; with no time intervening | ('now', 'immediately', 'directly', 'instantly')
- synonyms | and nothing more | ('but', 'just', 'only', 'simply')
- synonyms | slang for sexual intercourse | ('fuck', 'fucking', 'ass', 'screw')
- synonyms | show respect towards | ('respect', 'honor', 'honour', 'observe')


In [ ]:
from itertools import combinations

COLOR_SET = set([c.lower() for c in COLOR_WORDS])

def is_color_group(ws):
    return all(w.lower() in COLOR_SET for w in ws)

def shared_wordnet_hypernyms(ws, depth=3):
    # collect hypernyms for each word and intersect
    def hypers(word):
        out = set()
        for s in wn.synsets(word.lower()):
            frontier = [(s, 0)]
            seen = set([s])
            while frontier:
                node, d = frontier.pop()
                if d >= depth:
                    continue
                for h in node.hypernyms():
                    if h not in seen:
                        seen.add(h)
                        frontier.append((h, d+1))
                        out.add(h.name())
        return out

    sets = [hypers(w) for w in ws]
    inter = set.intersection(*sets) if sets else set()
    return inter

def looks_like_clean_category(ws):
    # colors
    if is_color_group(ws):
        return True

    # synonyms: if there exists a synset that contains all 4 as lemmas (rough)
    lemma_sets = []
    for w in ws:
        syns = wn.synsets(w.lower())
        lemma_sets.append(set(s.name() for s in syns))
    # if they share at least one synset name, it's often synonym-ish (strict)
    if set.intersection(*lemma_sets):
        return True

    # types-of: shared hypernym overlap (looser)
    inter_h = shared_wordnet_hypernyms(ws, depth=3)
    if len(inter_h) > 0:
        return True

    return False

def puzzle_is_ambiguous(words, intended_groups):
    """
    intended_groups: list of 4 tuples of words (the 4 categories you chose)
    """
    intended = set(frozenset(g) for g in intended_groups)
    wlist = list(words)

    for combo in combinations(wlist, 4):
        f = frozenset(combo)
        if f in intended:
            continue
        if looks_like_clean_category(combo):
            return True
    return False

def generate_nonambiguous_puzzle(bank, max_tries=80000, seed=7):
    random.seed(seed)
    for _ in range(max_tries):
        chosen = random.sample(bank, 4)
        groups = [tuple(c["words"]) for c in chosen]
        words = [w for c in chosen for w in c["words"]]
        if len(set(w.lower() for w in words)) != 16:
            continue
        if puzzle_is_ambiguous(words, groups):
            continue
        return chosen, words
    return None, None

chosen_cats, puzzle_words = generate_nonambiguous_puzzle(category_bank_clean, seed=7)

if chosen_cats is None:
    print("Could not find a non-ambiguous puzzle under current rules. Relax filters or increase bank.")
else:
    print("Puzzle words:", puzzle_words)
    print("\nCategories:")
    for c in chosen_cats:
        print("-", c["type"], "|", c["label"], "|", c["words"])

Puzzle words: ['now', 'immediately', 'directly', 'instantly', 'but', 'just', 'only', 'simply', 'fuck', 'fucking', 'ass', 'screw', 'respect', 'honor', 'honour', 'observe']

Categories:
- synonyms | without delay or hesitation; with no time intervening | ('now', 'immediately', 'directly', 'instantly')
- synonyms | and nothing more | ('but', 'just', 'only', 'simply')
- synonyms | slang for sexual intercourse | ('fuck', 'fucking', 'ass', 'screw')
- synonyms | show respect towards | ('respect', 'honor', 'honour', 'observe')


In [ ]:
chosen_cats2, puzzle_words2 = generate_puzzle(category_bank)
print("Puzzle words:", puzzle_words2)
print("\nCategories:")
for c in chosen_cats2:
    print("-", c["type"], "|", c["label"], "|", c["words"])


Puzzle words: None

Categories:


TypeError: 'NoneType' object is not iterable

In [ ]:
!pip -q install pandas numpy nltk wordfreq

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn

from wordfreq import zipf_frequency

In [ ]:
JSON_PATH = "connections.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(type(data), len(data))
print(data[0].keys())
print(data[0]["answers"][0].keys())

<class 'list'> 992
dict_keys(['id', 'date', 'answers'])
dict_keys(['level', 'group', 'members'])


In [ ]:
def extract_green_level1(data):
    rows = []

    for puzzle in data:
        puzzle_id = puzzle.get("id")
        date = puzzle.get("date")

        for ans in puzzle.get("answers", []):
            if ans.get("level") == 1:   # green
                members = ans.get("members", [])
                if len(members) != 4:
                    continue

                rows.append({
                    "id": puzzle_id,
                    "date": date,
                    "group": ans.get("group"),
                    "w1": members[0],
                    "w2": members[1],
                    "w3": members[2],
                    "w4": members[3],
                    "words": tuple(members)
                })

    return pd.DataFrame(rows)

green_df = extract_green_level1(data)
print("num green categories:", len(green_df))
green_df.head()

num green categories: 831


,id,date,group,w1,w2,w3,w4,words
0,1,2023-06-12,NBA TEAMS,BUCKS,HEAT,JAZZ,NETS,"(BUCKS, HEAT, JAZZ, NETS)"
1,2,2023-06-13,UNITS OF LENGTH,FOOT,LEAGUE,MILE,YARD,"(FOOT, LEAGUE, MILE, YARD)"
2,3,2023-06-14,SYNONYMS FOR EAT,CHOW,GOBBLE,SCARF,WOLF,"(CHOW, GOBBLE, SCARF, WOLF)"
3,4,2023-06-15,MUSICALS BEGINNING WITH “C”,CABARET,CAROUSEL,CATS,CHICAGO,"(CABARET, CAROUSEL, CATS, CHICAGO)"
4,5,2023-06-16,CONDIMENTS,KETCHUP,MAYO,RELISH,TARTAR,"(KETCHUP, MAYO, RELISH, TARTAR)"


In [ ]:
def polysemy_count(word):
    return len(wn.synsets(str(word).lower()))

def zipf_stats(words):
    vals = [zipf_frequency(str(w).lower(), "en") for w in words]
    return np.mean(vals), np.min(vals), np.max(vals)

def polysemy_stats(words):
    vals = [polysemy_count(w) for w in words]
    return np.mean(vals), np.min(vals), np.max(vals)

zipf_mean, zipf_min, zipf_max = [], [], []
poly_mean, poly_min, poly_max = [], [], []

for words in green_df["words"]:
    a, b, c = zipf_stats(words)
    zipf_mean.append(a)
    zipf_min.append(b)
    zipf_max.append(c)

    a, b, c = polysemy_stats(words)
    poly_mean.append(a)
    poly_min.append(b)
    poly_max.append(c)

green_df["zipf_mean"] = zipf_mean
green_df["zipf_min"] = zipf_min
green_df["zipf_max"] = zipf_max

green_df["poly_mean"] = poly_mean
green_df["poly_min"] = poly_min
green_df["poly_max"] = poly_max

green_df[["group", "words", "zipf_mean", "poly_mean"]].head(10)

,group,words,zipf_mean,poly_mean
0,NBA TEAMS,"(BUCKS, HEAT, JAZZ, NETS)",4.2925,8.75
1,UNITS OF LENGTH,"(FOOT, LEAGUE, MILE, YARD)",4.7900,8.75
2,SYNONYMS FOR EAT,"(CHOW, GOBBLE, SCARF, WOLF)",3.5725,4.25
3,MUSICALS BEGINNING WITH “C”,"(CABARET, CAROUSEL, CATS, CHICAGO)",3.9350,4.00
4,CONDIMENTS,"(KETCHUP, MAYO, RELISH, TARTAR)",3.3450,2.50
5,SHADES OF BLUE,"(BABY, MIDNIGHT, POWDER, ROYAL)",4.6675,5.25
6,BABY ANIMALS,"(CALF, CUB, JOEY, KID)",4.0275,3.75
7,MATTRESS SIZES,"(FULL, KING, QUEEN, TWIN)",4.9775,11.00
8,TOPS,"(CAMI, HALTER, TANK, TEE)",3.4450,4.75
9,COUNTRIES,"(CHAD, GEORGIA, JORDAN, TOGO)",3.9750,2.50


In [ ]:
green_df[["zipf_mean", "zipf_min", "poly_mean", "poly_max"]].describe()

,zipf_mean,zipf_min,poly_mean,poly_max
count,831.000000,831.000000,831.000000,831.000000
mean,4.184179,3.465211,8.146811,15.235860
std,0.652311,0.782054,5.204503,10.815861
min,0.000000,0.000000,0.000000,0.000000
25%,3.781250,3.000000,4.500000,8.000000
50%,4.210000,3.520000,7.250000,12.000000
75%,4.616250,3.970000,10.750000,19.500000
max,6.770000,6.500000,36.750000,75.000000


In [ ]:
green_df["auto_structure"] = ""
green_df["manual_structure"] = ""
green_df["notes"] = ""

In [ ]:
def auto_structure_label(group_name):
    g = str(group_name or "").strip().lower()

    if not g:
        return "other"

    # phrase-like or pattern-like categories
    phrase_markers = [
        "ending", "starting", "beginning", "before", "after",
        "followed by", "preceded by", "homophones", "palindromes",
        "words before", "words after", "can come after", "can come before"
    ]
    if any(m in g for m in phrase_markers):
        return "phrase_template"

    # place-based
    place_markers = [
        "found in", "found at", "seen in", "seen at",
        "places", "where you", "where one", "at a", "in a"
    ]
    if any(m in g for m in place_markers):
        return "place_based"

    # activity-based
    activity_markers = [
        "used for", "things you", "used when", "for cooking", "for sewing",
        "for writing", "for travel", "for sports"
    ]
    if any(m in g for m in activity_markers):
        return "activity_based"

    # synonyms
    synonym_markers = [
        "synonyms", "another word for", "means", "ways to say"
    ]
    if any(m in g for m in synonym_markers):
        return "synonyms"

    # attribute/property
    attr_markers = [
        "colors", "shades", "materials", "parts of", "units of",
        "types of", "kinds of"
    ]
    if any(m in g for m in attr_markers):
        return "attribute_based"

    # strict ontological categories
    strict_markers = [
        "teams", "fruits", "vegetables", "flowers", "birds", "animals",
        "magazines", "keys", "shoes", "footwear", "cars", "states",
        "countries", "planets", "tools", "instruments"
    ]
    if any(m in g for m in strict_markers):
        return "same_type_strict"

    return "same_type_everyday"

green_df["auto_structure"] = green_df["group"].map(auto_structure_label)
green_df[["group", "auto_structure"]].head(20)

,group,auto_structure
0,NBA TEAMS,same_type_strict
1,UNITS OF LENGTH,attribute_based
2,SYNONYMS FOR EAT,synonyms
3,MUSICALS BEGINNING WITH “C”,phrase_template
4,CONDIMENTS,same_type_everyday
5,SHADES OF BLUE,attribute_based
6,BABY ANIMALS,same_type_strict
7,MATTRESS SIZES,same_type_everyday
8,TOPS,same_type_everyday
9,COUNTRIES,same_type_strict


In [ ]:
review_cols = [
    "id", "date", "group", "w1", "w2", "w3", "w4",
    "auto_structure", "manual_structure", "notes"
]

review_df = green_df[review_cols].copy()

OUT_PATH = "/content/green_review.csv"
review_df.to_csv(OUT_PATH, index=False)

print("saved:", OUT_PATH)
review_df.head()

saved: /content/green_review.csv


,id,date,group,w1,w2,w3,w4,auto_structure,manual_structure,notes
0,1,2023-06-12,NBA TEAMS,BUCKS,HEAT,JAZZ,NETS,,,
1,2,2023-06-13,UNITS OF LENGTH,FOOT,LEAGUE,MILE,YARD,,,
2,3,2023-06-14,SYNONYMS FOR EAT,CHOW,GOBBLE,SCARF,WOLF,,,
3,4,2023-06-15,MUSICALS BEGINNING WITH “C”,CABARET,CAROUSEL,CATS,CHICAGO,,,
4,5,2023-06-16,CONDIMENTS,KETCHUP,MAYO,RELISH,TARTAR,,,


In [ ]:
labeled_df = pd.read_csv("/content/green_review_labeled.csv")

labeled_df["final_structure"] = labeled_df["manual_structure"].fillna("").replace("", np.nan)
labeled_df["final_structure"] = labeled_df["final_structure"].fillna(labeled_df["auto_structure"])

print(labeled_df["final_structure"].value_counts())
print()
print((labeled_df["final_structure"].value_counts(normalize=True) * 100).round(1))

final_structure
same_type_everyday    637
strict_class           43
attribute_based        39
place_based            28
synonym_cluster        23
property_category      14
phrase_template        11
same_type_strict       11
domain_class            8
activity_based          6
object_parts            4
linguistic_pattern      4
phrase_association      2
synonyms                1
Name: count, dtype: int64

final_structure
same_type_everyday    76.7
strict_class           5.2
attribute_based        4.7
place_based            3.4
synonym_cluster        2.8
property_category      1.7
phrase_template        1.3
same_type_strict       1.3
domain_class           1.0
activity_based         0.7
object_parts           0.5
linguistic_pattern     0.5
phrase_association     0.2
synonyms               0.1
Name: proportion, dtype: float64


In [ ]:
def tokenize_group(text):
    text = str(text or "").lower()
    toks = re.findall(r"[a-z]+", text)
    stop = {
        "a","an","the","of","in","at","to","for","and","or","with",
        "things","words","items","that","you","one"
    }
    return [t for t in toks if t not in stop]

token_counts = Counter()
for g in green_df["group"]:
    token_counts.update(tokenize_group(g))

token_counts.most_common(50)

[('as', 24),
 ('kinds', 23),
 ('parts', 15),
 ('s', 14),
 ('on', 13),
 ('do', 12),
 ('features', 9),
 ('up', 9),
 ('terms', 8),
 ('ways', 8),
 ('used', 8),
 ('slang', 7),
 ('are', 7),
 ('objects', 7),
 ('seen', 7),
 ('units', 6),
 ('sports', 6),
 ('musical', 6),
 ('bit', 6),
 ('hair', 6),
 ('move', 6),
 ('some', 6),
 ('car', 6),
 ('elements', 5),
 ('small', 5),
 ('instruments', 5),
 ('info', 5),
 ('adjectives', 5),
 ('from', 5),
 ('animals', 4),
 ('shapes', 4),
 ('board', 4),
 ('games', 4),
 ('types', 4),
 ('sections', 4),
 ('golf', 4),
 ('food', 4),
 ('go', 4),
 ('time', 4),
 ('amount', 4),
 ('prefixes', 4),
 ('can', 4),
 ('into', 4),
 ('might', 4),
 ('mess', 4),
 ('green', 4),
 ('equipment', 4),
 ('it', 4),
 ('be', 4),
 ('needs', 4)]

In [ ]:
green_df["group"].value_counts().head(40)

,count
group,
MESS OF HAIR,2
CABLE CHANNELS,2
CARD GAMES,2
SHADES OF GREEN,2
TREES,2
MILD OATHS,2
THINGS WITH SHELLS,2
INFLUENCE,2
MOVE QUICKLY,2


In [ ]:
structure_examples = defaultdict(list)

for _, row in labeled_df.iterrows():
    structure = row["final_structure"]
    structure_examples[structure].append({
        "group": row["group"],
        "words": [row["w1"], row["w2"], row["w3"], row["w4"]]
    })

for structure, exs in structure_examples.items():
    print("\nSTRUCTURE:", structure, "| count:", len(exs))
    for ex in exs[:5]:
        print("  ", ex)


STRUCTURE: strict_class | count: 43
   {'group': 'NBA TEAMS', 'words': ['BUCKS', 'HEAT', 'JAZZ', 'NETS']}
   {'group': 'UNITS OF LENGTH', 'words': ['FOOT', 'LEAGUE', 'MILE', 'YARD']}
   {'group': 'CONDIMENTS', 'words': ['KETCHUP', 'MAYO', 'RELISH', 'TARTAR']}
   {'group': 'MATTRESS SIZES', 'words': ['FULL', 'KING', 'QUEEN', 'TWIN']}
   {'group': 'TOPS', 'words': ['CAMI', 'HALTER', 'TANK', 'TEE']}

STRUCTURE: synonym_cluster | count: 23
   {'group': 'SYNONYMS FOR EAT', 'words': ['CHOW', 'GOBBLE', 'SCARF', 'WOLF']}
   {'group': 'TERMS OF ENDEARMENT', 'words': ['BOO', 'HONEY', 'SUGAR', 'SWEETIE']}
   {'group': 'MILD OATHS', 'words': ['DARN', 'FUDGE', 'HECK', 'SHOOT']}
   {'group': 'SYNONYMS FOR IMITATE', 'words': ['COPY', 'ECHO', 'MIMIC', 'PARROT']}
   {'group': 'COVERINGS', 'words': ['CAP', 'COVER', 'LID', 'TOP']}

STRUCTURE: linguistic_pattern | count: 4
   {'group': 'MUSICALS BEGINNING WITH “C”', 'words': ['CABARET', 'CAROUSEL', 'CATS', 'CHICAGO']}
   {'group': 'WORDS WITH THREE G’S',

In [ ]:
def avg_word_len(words):
    return np.mean([len(str(w)) for w in words])

green_df["avg_word_len"] = green_df["words"].map(avg_word_len)

green_df[["avg_word_len", "zipf_mean", "zipf_min", "poly_mean", "poly_max"]].describe()

,avg_word_len,zipf_mean,zipf_min,poly_mean,poly_max
count,831.000000,831.000000,831.000000,831.000000,831.000000
mean,5.414561,4.184179,3.465211,8.146811,15.235860
std,1.332713,0.652311,0.782054,5.204503,10.815861
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.500000,3.781250,3.000000,4.500000,8.000000
50%,5.250000,4.210000,3.520000,7.250000,12.000000
75%,6.000000,4.616250,3.970000,10.750000,19.500000
max,12.250000,6.770000,6.500000,36.750000,75.000000


In [ ]:
# ============================================================
# STEP 11: Generate new GREEN-category candidates
# from your labeled green archive
#
# Assumes you already have:
#   labeled_df
# with columns:
#   group, w1, w2, w3, w4, final_structure
# ============================================================

!pip -q install nltk wordfreq gensim

import re
import random
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn

from wordfreq import zipf_frequency
from gensim import downloader as api

# smaller / faster than fastText, good enough for first pass
wv = api.load("glove-wiki-gigaword-100")

# ------------------------------------------------------------
# 1) helpers
# ------------------------------------------------------------
def norm_word(w):
    return str(w).strip().upper()

def clean_token(w):
    w = str(w).strip()
    return w

def word_ok(w, min_zipf=3.2, max_poly=12):
    """
    Basic filter for puzzle-suitable words.
    Allows spaces for entries like SCHOOL BUS.
    """
    w2 = w.strip()
    if not w2:
        return False
    if len(w2) < 2 or len(w2) > 20:
        return False
    if not re.fullmatch(r"[A-Za-z ]+", w2):
        return False
    # frequency check on each token
    toks = w2.lower().split()
    if min(zipf_frequency(t, "en") for t in toks) < min_zipf:
        return False
    # polysemy check only for single-token words
    if len(toks) == 1 and len(wn.synsets(toks[0])) > max_poly:
        return False
    return True

def group_words(row):
    return [row["w1"], row["w2"], row["w3"], row["w4"]]

def tokenize_group_label(text):
    text = str(text or "").upper()
    toks = re.findall(r"[A-Z]+", text)
    stop = {"OF", "FOR", "THE", "AND", "THAT", "ARE", "WITH", "IN", "AT", "TO"}
    return [t for t in toks if t not in stop]

def get_vec(word):
    """
    Average token vectors for multiword entries.
    """
    toks = word.lower().split()
    vecs = []
    for t in toks:
        if t in wv:
            vecs.append(wv[t])
    if not vecs:
        return None
    return np.mean(vecs, axis=0)

def cosine(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12)
    return float(np.dot(a, b) / denom)

def avg_pairwise_similarity(words):
    vecs = [get_vec(w) for w in words]
    vecs = [v for v in vecs if v is not None]
    if len(vecs) < 2:
        return -1
    sims = []
    for i in range(len(vecs)):
        for j in range(i+1, len(vecs)):
            sims.append(cosine(vecs[i], vecs[j]))
    return float(np.mean(sims)) if sims else -1

def is_plural_pair(a, b):
    a2, b2 = a.lower(), b.lower()
    return a2 == b2 + "s" or b2 == a2 + "s"

def too_similar_surface(a, b):
    a2, b2 = a.lower(), b.lower()
    if a2 == b2:
        return True
    if is_plural_pair(a2, b2):
        return True
    if a2[:4] == b2[:4] and len(a2) >= 5 and len(b2) >= 5:
        return True
    return False


# ------------------------------------------------------------
# 2) build archive buckets from labeled data
# ------------------------------------------------------------
required_cols = {"group", "w1", "w2", "w3", "w4", "final_structure"}
missing = required_cols - set(labeled_df.columns)
if missing:
    raise ValueError(f"labeled_df missing columns: {missing}")

archive = labeled_df.copy()
archive["words"] = archive.apply(group_words, axis=1)

structure_examples = defaultdict(list)
word_to_groups = defaultdict(set)
group_token_counts = Counter()

for _, row in archive.iterrows():
    s = row["final_structure"]
    g = row["group"]
    ws = [norm_word(w) for w in row["words"]]
    structure_examples[s].append({"group": g, "words": ws})
    for w in ws:
        word_to_groups[w].add(g)
    group_token_counts.update(tokenize_group_label(g))

print("Structures found:")
for k, v in sorted(structure_examples.items(), key=lambda x: len(x[1]), reverse=True):
    print(f"{k:20s} {len(v)}")


# ------------------------------------------------------------
# 3) seed lexicons from your archive
# ------------------------------------------------------------
# These become candidate pools for generation.

strict_like_words = set()
synonym_like_words = set()
property_like_words = set()
domain_like_words = set()
object_part_words = set()
phrase_assoc_words = set()

for s, examples in structure_examples.items():
    for ex in examples:
        ws = ex["words"]
        if s == "strict_class":
            strict_like_words.update(ws)
        elif s == "synonym_cluster":
            synonym_like_words.update(ws)
        elif s == "property_category":
            property_like_words.update(ws)
        elif s == "domain_class":
            domain_like_words.update(ws)
        elif s == "object_parts":
            object_part_words.update(ws)
        elif s == "phrase_association":
            phrase_assoc_words.update(ws)

print("strict pool:", len(strict_like_words))
print("synonym pool:", len(synonym_like_words))
print("property pool:", len(property_like_words))
print("domain pool:", len(domain_like_words))
print("parts pool:", len(object_part_words))
print("phrase pool:", len(phrase_assoc_words))


# ------------------------------------------------------------
# 4) structure distribution learned from archive
# ------------------------------------------------------------
structure_probs = (
    archive["final_structure"]
    .value_counts(normalize=True)
    .to_dict()
)

print("\nLearned structure distribution:")
for k, v in structure_probs.items():
    print(f"{k:20s} {v:.3f}")


# ------------------------------------------------------------
# 5) generators by structure type
# ------------------------------------------------------------

# ---------- strict class ----------
def strict_from_wordnet(max_candidates=50):
    """
    Build strict categories like BIRDS / FRUITS / TOOLS / etc.
    """
    cands = []
    noun_synsets = list(wn.all_synsets(pos=wn.NOUN))
    random.shuffle(noun_synsets)

    for parent in noun_synsets:
        parent_names = [l.name().replace("_", " ").upper() for l in parent.lemmas()]
        parent_names = [p for p in parent_names if word_ok(p)]
        if not parent_names:
            continue

        hypos = set(parent.hyponyms())
        for h in list(hypos):
            hypos.update(h.hyponyms())

        words = []
        for h in hypos:
            for l in h.lemmas():
                w = l.name().replace("_", " ").upper()
                if word_ok(w):
                    words.append(w)

        # unique, common-first
        words = list(dict.fromkeys(words))
        words = sorted(words, key=lambda w: -min(zipf_frequency(t.lower(), "en") for t in w.split()))

        # diversity cleanup
        chosen = []
        for w in words:
            if any(too_similar_surface(w, x) for x in chosen):
                continue
            chosen.append(w)
            if len(chosen) == 4:
                break

        if len(chosen) == 4:
            label = parent_names[0]
            cands.append({
                "structure": "strict_class",
                "group": label,
                "words": chosen,
                "source": "wordnet_strict"
            })

        if len(cands) >= max_candidates:
            break

    return cands


# ---------- synonym cluster ----------
def synonym_candidates(max_candidates=50):
    cands = []
    synsets = list(wn.all_synsets())
    random.shuffle(synsets)

    for syn in synsets:
        lemmas = [l.name().replace("_", " ").upper() for l in syn.lemmas()]
        lemmas = [w for w in lemmas if word_ok(w)]
        lemmas = list(dict.fromkeys(lemmas))
        lemmas = sorted(lemmas, key=lambda w: -min(zipf_frequency(t.lower(), "en") for t in w.split()))

        chosen = []
        for w in lemmas:
            if any(too_similar_surface(w, x) for x in chosen):
                continue
            chosen.append(w)
            if len(chosen) == 4:
                break

        if len(chosen) == 4:
            head = chosen[0]
            cands.append({
                "structure": "synonym_cluster",
                "group": f"SYNONYMS FOR {head}",
                "words": chosen,
                "source": "wordnet_synonyms"
            })

        if len(cands) >= max_candidates:
            break

    return cands


# ---------- property category ----------
PROPERTY_TEMPLATES = [
    ("THINGS THAT ARE RED", ["APPLE", "FIRE TRUCK", "ROSE", "STOP SIGN"]),
    ("THINGS THAT ARE ROUND", ["BALL", "COIN", "MOON", "WHEEL"]),
    ("THINGS THAT ARE COLD", ["ICE", "SNOW", "FREEZER", "POPSICLE"]),
    ("THINGS THAT ARE SWEET", ["CANDY", "HONEY", "SYRUP", "CAKE"]),
    ("THINGS THAT ARE LOUD", ["ALARM", "SIREN", "THUNDER", "DRUM"]),
]

def property_candidates(max_candidates=20):
    cands = []
    for label, words in PROPERTY_TEMPLATES[:max_candidates]:
        words = [w for w in words if word_ok(w)]
        if len(words) == 4:
            cands.append({
                "structure": "property_category",
                "group": label,
                "words": words,
                "source": "seed_property"
            })
    return cands


# ---------- domain class ----------
DOMAIN_TEMPLATES = [
    ("PROGRAMMING LANGUAGES", ["JAVA", "PYTHON", "RUBY", "SWIFT"]),
    ("SPREADSHEET FUNCTIONS", ["AVERAGE", "COUNT", "SUM", "VLOOKUP"]),
    ("FILE TYPES", ["CSV", "DOCX", "HTML", "PDF"]),
    ("MATH SYMBOLS", ["DELTA", "PI", "SIGMA", "THETA"]),
    ("CHESS PIECES", ["BISHOP", "KING", "KNIGHT", "ROOK"]),
]

def domain_candidates(max_candidates=20):
    cands = []
    for label, words in DOMAIN_TEMPLATES[:max_candidates]:
        words = [w for w in words if word_ok(w)]
        if len(words) == 4:
            cands.append({
                "structure": "domain_class",
                "group": label,
                "words": words,
                "source": "seed_domain"
            })
    return cands


# ---------- object parts ----------
PART_TEMPLATES = [
    ("CAR PARTS", ["AXLE", "BRAKE", "DOOR", "HOOD"]),
    ("FACE PARTS", ["CHEEK", "CHIN", "LIP", "NOSE"]),
    ("COMPUTER PARTS", ["MOUSE", "MONITOR", "SCREEN", "KEYBOARD"]),
    ("BICYCLE PARTS", ["CHAIN", "PEDAL", "SADDLE", "SPOKE"]),
]

def object_part_candidates(max_candidates=20):
    cands = []
    for label, words in PART_TEMPLATES[:max_candidates]:
        words = [w for w in words if word_ok(w)]
        if len(words) == 4:
            cands.append({
                "structure": "object_parts",
                "group": label,
                "words": words,
                "source": "seed_parts"
            })
    return cands


# ---------- phrase association ----------
PHRASE_TEMPLATES = [
    ("THINGS FOUND IN A GARDEN", ["FLOWER", "HOSE", "SHOVEL", "WEED"]),
    ("THINGS FOUND IN A WALLET", ["CASH", "CARD", "ID", "RECEIPT"]),
    ("THINGS SEEN AT A BEACH", ["SAND", "SHELL", "SURFBOARD", "UMBRELLA"]),
    ("THINGS FOUND IN A CLASSROOM", ["CHALK", "DESK", "ERASER", "MARKER"]),
    ("THINGS FOUND IN A KITCHEN", ["KNIFE", "PAN", "SPONGE", "WHISK"]),
]

def phrase_assoc_candidates(max_candidates=20):
    cands = []
    for label, words in PHRASE_TEMPLATES[:max_candidates]:
        words = [w for w in words if word_ok(w)]
        if len(words) == 4:
            cands.append({
                "structure": "phrase_association",
                "group": label,
                "words": words,
                "source": "seed_phrase"
            })
    return cands


# ------------------------------------------------------------
# 6) combine candidate pools
# ------------------------------------------------------------
candidate_bank = []
candidate_bank.extend(strict_from_wordnet(max_candidates=120))
candidate_bank.extend(synonym_candidates(max_candidates=80))
candidate_bank.extend(property_candidates(max_candidates=20))
candidate_bank.extend(domain_candidates(max_candidates=20))
candidate_bank.extend(object_part_candidates(max_candidates=20))
candidate_bank.extend(phrase_assoc_candidates(max_candidates=20))

print("Total raw candidate groups:", len(candidate_bank))


# ------------------------------------------------------------
# 7) score candidates to feel more like GREEN
# ------------------------------------------------------------
# We use your archive stats as a target profile.

target_zipf_mean = archive["words"].apply(
    lambda ws: np.mean([zipf_frequency(str(w).lower().split()[0], "en") for w in ws])
).mean()

target_len_mean = archive["words"].apply(
    lambda ws: np.mean([len(str(w)) for w in ws])
).mean()

def score_candidate(c):
    words = c["words"]

    # commonness
    zipf_vals = []
    for w in words:
        toks = w.lower().split()
        zipf_vals.append(np.mean([zipf_frequency(t, "en") for t in toks]))
    zipf_mean = float(np.mean(zipf_vals))

    # surface simplicity
    len_mean = float(np.mean([len(w) for w in words]))

    # semantic cohesion
    coh = avg_pairwise_similarity(words)

    # penalties
    dup_pen = 0
    for i in range(4):
        for j in range(i+1, 4):
            if too_similar_surface(words[i], words[j]):
                dup_pen += 1

    # closer to archive style = better
    score = 0.0
    score += 1.8 * coh
    score -= 0.25 * abs(zipf_mean - target_zipf_mean)
    score -= 0.10 * abs(len_mean - target_len_mean)
    score -= 0.75 * dup_pen

    return {
        **c,
        "cohesion": coh,
        "zipf_mean": zipf_mean,
        "len_mean": len_mean,
        "score": score
    }

scored_candidates = [score_candidate(c) for c in candidate_bank]
scored_candidates = sorted(scored_candidates, key=lambda x: x["score"], reverse=True)

pd.DataFrame(scored_candidates)[["structure", "group", "words", "source", "score", "cohesion"]].head(20)

[==================================================] 100.0% 128.1/128.1MB downloaded
Structures found:
same_type_everyday   637
strict_class         43
attribute_based      39
place_based          28
synonym_cluster      23
property_category    14
same_type_strict     11
phrase_template      11
domain_class         8
activity_based       6
linguistic_pattern   4
object_parts         4
phrase_association   2
synonyms             1
strict pool: 171
synonym pool: 91
property pool: 55
domain pool: 32
parts pool: 16
phrase pool: 8

Learned structure distribution:
same_type_everyday   0.767
strict_class         0.052
attribute_based      0.047
place_based          0.034
synonym_cluster      0.028
property_category    0.017
phrase_template      0.013
same_type_strict     0.013
domain_class         0.010
activity_based       0.007
object_parts         0.005
linguistic_pattern   0.005
phrase_association   0.002
synonyms             0.001
Total raw candidate groups: 206


,structure,group,words,source,score,cohesion
0,synonym_cluster,SYNONYMS FOR TERRIBLE,"[TERRIBLE, AWFUL, PAINFUL, DREADFUL]",wordnet_synonyms,1.124172,0.724728
1,strict_class,LIMB,"[ARM, ANIMAL LEG, LEG, BOW LEG]",wordnet_strict,1.123924,0.705667
2,property_category,THINGS THAT ARE SWEET,"[CANDY, HONEY, SYRUP, CAKE]",seed_property,1.057616,0.630049
3,strict_class,WATER TRAVEL,"[OCEAN TRIP, CRUISE, SAIL, SAILING]",wordnet_strict,0.990062,0.631820
4,strict_class,PIPE,"[WATER MAIN, MAIN, GAS MAIN, FUEL LINE]",wordnet_strict,0.948023,0.797006
5,strict_class,STAR,"[TV STAR, FILM STAR, MOVIE STAR, TELEVISION STAR]",wordnet_strict,0.932220,0.916873
6,strict_class,BROADCASTING STATION,"[RADIO STATION, TV STATION, CHANNEL, TV CHANNEL]",wordnet_strict,0.912733,0.872540
7,strict_class,WATER SPORT,"[SWIMMING, SWIM, DIVE, SKIN DIVING]",wordnet_strict,0.897117,0.587301
8,synonym_cluster,SYNONYMS FOR GET OFF,"[GET OFF, TURN ON, TRIP, TRIP OUT]",wordnet_synonyms,0.888337,0.760723
9,strict_class,FRACTION,"[HALF, THIRD, ONE PERCENT, TEN PERCENT]",wordnet_strict,0.882377,0.784147


In [ ]:
# ============================================================
# STEP 12: Deduplicate and sample candidates in the same style
# ============================================================

def canonical_words(words):
    return tuple(sorted(norm_word(w) for w in words))

def dedupe_candidates(cands):
    seen_wordsets = set()
    seen_groups = set()
    out = []

    for c in cands:
        key_words = canonical_words(c["words"])
        key_group = str(c["group"]).upper()

        if key_words in seen_wordsets:
            continue
        if key_group in seen_groups:
            continue

        seen_wordsets.add(key_words)
        seen_groups.add(key_group)
        out.append(c)

    return out

final_candidates = dedupe_candidates(scored_candidates)
print("After dedupe:", len(final_candidates))

cand_df = pd.DataFrame(final_candidates)
cand_df[["structure", "group", "words", "score", "source"]].head(30)

After dedupe: 202


,structure,group,words,score,source
0,synonym_cluster,SYNONYMS FOR TERRIBLE,"[TERRIBLE, AWFUL, PAINFUL, DREADFUL]",1.124172,wordnet_synonyms
1,strict_class,LIMB,"[ARM, ANIMAL LEG, LEG, BOW LEG]",1.123924,wordnet_strict
2,property_category,THINGS THAT ARE SWEET,"[CANDY, HONEY, SYRUP, CAKE]",1.057616,seed_property
3,strict_class,WATER TRAVEL,"[OCEAN TRIP, CRUISE, SAIL, SAILING]",0.990062,wordnet_strict
4,strict_class,PIPE,"[WATER MAIN, MAIN, GAS MAIN, FUEL LINE]",0.948023,wordnet_strict
5,strict_class,STAR,"[TV STAR, FILM STAR, MOVIE STAR, TELEVISION STAR]",0.932220,wordnet_strict
6,strict_class,BROADCASTING STATION,"[RADIO STATION, TV STATION, CHANNEL, TV CHANNEL]",0.912733,wordnet_strict
7,strict_class,WATER SPORT,"[SWIMMING, SWIM, DIVE, SKIN DIVING]",0.897117,wordnet_strict
8,synonym_cluster,SYNONYMS FOR GET OFF,"[GET OFF, TURN ON, TRIP, TRIP OUT]",0.888337,wordnet_synonyms
9,strict_class,FRACTION,"[HALF, THIRD, ONE PERCENT, TEN PERCENT]",0.882377,wordnet_strict


In [ ]:


def weighted_structure_sample(structure_probs):
    keys = list(structure_probs.keys())
    vals = np.array([structure_probs[k] for k in keys], dtype=float)
    vals = vals / vals.sum()
    return np.random.choice(keys, p=vals)

def generate_green_batch(cands, structure_probs, n=20, seed=7):
    random.seed(seed)
    np.random.seed(seed)

    by_structure = defaultdict(list)
    for c in cands:
        by_structure[c["structure"]].append(c)

    # sort within each structure by score
    for s in by_structure:
        by_structure[s] = sorted(by_structure[s], key=lambda x: x["score"], reverse=True)

    chosen = []
    used_wordsets = set()
    used_groups = set()

    tries = 0
    while len(chosen) < n and tries < 5000:
        tries += 1
        s = weighted_structure_sample(structure_probs)

        if s not in by_structure or not by_structure[s]:
            continue

        # bias toward better candidates
        bucket = by_structure[s][: min(40, len(by_structure[s]))]
        c = random.choice(bucket)

        words_key = canonical_words(c["words"])
        group_key = c["group"].upper()

        if words_key in used_wordsets or group_key in used_groups:
            continue

        used_wordsets.add(words_key)
        used_groups.add(group_key)
        chosen.append(c)

    return pd.DataFrame(chosen)

generated_green_df = generate_green_batch(final_candidates, structure_probs, n=25, seed=11)
generated_green_df[["structure", "group", "words", "score", "source"]]

,structure,group,words,score,source
0,property_category,THINGS THAT ARE LOUD,"[ALARM, SIREN, THUNDER, DRUM]",0.546327,seed_property
1,strict_class,GREETING CARD,"[CHRISTMAS CARD, BIRTHDAY CARD, EASTER CARD, V...",0.575763,wordnet_strict
2,strict_class,ORE,"[PAY DIRT, IRON ORE, DRESSED ORE, LEAD ORE]",0.623723,wordnet_strict
3,strict_class,GLASS,"[LEAD GLASS, NATURAL GLASS, SAFETY GLASS, GROU...",0.624750,wordnet_strict
4,synonym_cluster,SYNONYMS FOR POINT OUT,"[POINT OUT, NOTICE, COMMENT, REMARK]",0.552075,wordnet_synonyms
5,strict_class,LIVING QUARTERS,"[FIRST CLASS, SECOND CLASS, THIRD CLASS, GUN R...",0.557516,wordnet_strict
6,strict_class,PEA,"[GREEN PEA, GARDEN PEA, SUGAR SNAP PEA, FIELD ...",0.792114,wordnet_strict
7,property_category,THINGS THAT ARE SWEET,"[CANDY, HONEY, SYRUP, CAKE]",1.057616,seed_property
8,strict_class,WEALTH,"[MONEY, BIG MONEY, PILE, BUNDLE]",0.598423,wordnet_strict
9,strict_class,GOOSE,"[CHINESE GOOSE, SNOW GOOSE, CANADA GOOSE, BLUE...",0.623632,wordnet_strict


In [ ]:

out = generated_green_df.copy()
out["w1"] = out["words"].map(lambda x: x[0])
out["w2"] = out["words"].map(lambda x: x[1])
out["w3"] = out["words"].map(lambda x: x[2])
out["w4"] = out["words"].map(lambda x: x[3])

review_cols = ["structure", "group", "w1", "w2", "w3", "w4", "score", "source"]
review_out = out[review_cols].copy()

OUT_PATH = "/content/generated_green_candidates.csv"
review_out.to_csv(OUT_PATH, index=False)

print("saved:", OUT_PATH)
review_out.head(20)

saved: /content/generated_green_candidates.csv


,structure,group,w1,w2,w3,w4,score,source
0,property_category,THINGS THAT ARE LOUD,ALARM,SIREN,THUNDER,DRUM,0.546327,seed_property
1,strict_class,GREETING CARD,CHRISTMAS CARD,BIRTHDAY CARD,EASTER CARD,VALENTINE,0.575763,wordnet_strict
2,strict_class,ORE,PAY DIRT,IRON ORE,DRESSED ORE,LEAD ORE,0.623723,wordnet_strict
3,strict_class,GLASS,LEAD GLASS,NATURAL GLASS,SAFETY GLASS,GROUND GLASS,0.624750,wordnet_strict
4,synonym_cluster,SYNONYMS FOR POINT OUT,POINT OUT,NOTICE,COMMENT,REMARK,0.552075,wordnet_synonyms
5,strict_class,LIVING QUARTERS,FIRST CLASS,SECOND CLASS,THIRD CLASS,GUN ROOM,0.557516,wordnet_strict
6,strict_class,PEA,GREEN PEA,GARDEN PEA,SUGAR SNAP PEA,FIELD PEA,0.792114,wordnet_strict
7,property_category,THINGS THAT ARE SWEET,CANDY,HONEY,SYRUP,CAKE,1.057616,seed_property
8,strict_class,WEALTH,MONEY,BIG MONEY,PILE,BUNDLE,0.598423,wordnet_strict
9,strict_class,GOOSE,CHINESE GOOSE,SNOW GOOSE,CANADA GOOSE,BLUE GOOSE,0.623632,wordnet_strict


In [ ]:
df = generated_green_df.copy()
df = df.sort_values(by='score', ascending=False)
df = df[df['source'] != 'wordnet_synonyms']
df.head(10)

,structure,group,words,source,cohesion,zipf_mean,len_mean,score
7,property_category,THINGS THAT ARE SWEET,"[CANDY, HONEY, SYRUP, CAKE]",seed_property,0.630049,4.23750,4.75,1.057616
20,strict_class,BROADCASTING STATION,"[TV STATION, RADIO STATION, CHANNEL, TV CHANNEL]",wordnet_strict,0.872540,4.99750,10.00,0.912733
19,strict_class,PIPE,"[MAIN, WATER MAIN, GAS MAIN, ELECTRIC MAIN]",wordnet_strict,0.825035,5.18125,8.75,0.906286
11,strict_class,WORD,"[PART NAME, WHOLE NAME, HEAD WORD, WORD FORM]",wordnet_strict,0.831550,5.47500,9.25,0.794577
6,strict_class,JET ENGINE,"[NUCLEAR ROCKET, ROCKET, SPACE ROCKET, BOOSTER]",wordnet_strict,0.683548,4.29000,9.75,0.774424
22,strict_class,DOG,"[POLICE DOG, WORKING DOG, COACH DOG, GUN DOG]",wordnet_strict,0.740579,5.13750,9.25,0.715204
3,strict_class,BIRD,"[SEA BIRD, FLYING BIRD, WATER BIRD, SECRETARY ...",wordnet_strict,0.746353,4.84500,10.75,0.648722
2,strict_class,GOAL,"[WILL, THING, BUSINESS, IDEA]",wordnet_strict,0.585133,5.77250,5.25,0.643017
9,strict_class,ORE,"[PAY DIRT, DRESSED ORE, LEAD ORE, IRON ORE]",wordnet_strict,0.578645,4.53750,8.75,0.623723
8,strict_class,DWELLING,"[COUNTRY HOUSE, TOWN HOUSE, SAFE HOUSE, BEACH ...",wordnet_strict,0.828205,5.44375,11.00,0.621369


In [ ]:
import re
import random
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

from nltk.corpus import wordnet as wn
from wordfreq import zipf_frequency

# ---------------------------
# normalization
# ---------------------------
def norm_word(w):
    return str(w).strip().upper()

def norm_label(x):
    return re.sub(r"\s+", " ", str(x).strip().upper())

def alpha_space_only(x):
    return bool(re.fullmatch(r"[A-Z ]+", norm_label(x)))

def token_zipf(word):
    toks = word.lower().split()
    if not toks:
        return 0.0
    return min(zipf_frequency(t, "en") for t in toks)

def avg_token_zipf(words):
    vals = [token_zipf(w) for w in words]
    return float(np.mean(vals)) if vals else 0.0

def polysemy_count(word):
    toks = word.lower().split()
    if len(toks) != 1:
        return 0
    return len(wn.synsets(toks[0]))

def word_ok_general(w, min_zipf=3.6, max_poly=10, min_len=3, max_len=16):
    w = norm_word(w)
    if not alpha_space_only(w):
        return False
    if len(w) < min_len or len(w) > max_len:
        return False
    if token_zipf(w) < min_zipf:
        return False
    if len(w.split()) == 1 and polysemy_count(w) > max_poly:
        return False
    return True

def word_ok_strict(w):
    return word_ok_general(w, min_zipf=4.0, max_poly=9, min_len=3, max_len=15)

def word_ok_synonym(w):
    return word_ok_general(w, min_zipf=3.8, max_poly=8, min_len=3, max_len=14)

# ---------------------------
# surface repetition blockers
# ---------------------------
def crude_stem(w):
    w = w.lower()
    for suf in ["ing", "ers", "er", "ed", "es", "s"]:
        if w.endswith(suf) and len(w) > len(suf) + 2:
            return w[:-len(suf)]
    return w

def too_similar_surface(a, b):
    a2 = a.lower()
    b2 = b.lower()

    if a2 == b2:
        return True

    if crude_stem(a2) == crude_stem(b2):
        return True

    if a2.replace(" ", "") == b2.replace(" ", ""):
        return True

    # same long prefix
    if len(a2) >= 5 and len(b2) >= 5 and a2[:4] == b2[:4]:
        return True

    # same long suffix
    if len(a2) >= 5 and len(b2) >= 5 and a2[-4:] == b2[-4:]:
        return True

    return False

def group_has_surface_repetition(words):
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            if too_similar_surface(words[i], words[j]):
                return True
    return False

# ---------------------------
# label quality
# ---------------------------
def label_quality(label):
    lab = norm_label(label)
    toks = lab.split()
    score = 0.0

    if 1 <= len(toks) <= 3:
        score += 1.0
    if all(zipf_frequency(t.lower(), "en") >= 3.8 for t in toks):
        score += 1.0
    if len(lab) <= 22:
        score += 0.5
    if lab.endswith("S"):
        score += 0.25

    bad_bits = ["ENTITY", "OBJECT", "ARTIFACT", "PHYSICAL", "WHOLE", "UNIT"]
    if any(x in lab for x in bad_bits):
        score -= 2.0

    return score

In [ ]:
# assumes labeled_df has columns:
# group, w1, w2, w3, w4, final_structure

archive = labeled_df.copy()
archive["group"] = archive["group"].map(norm_label)
for c in ["w1", "w2", "w3", "w4"]:
    archive[c] = archive[c].map(norm_word)

archive["words"] = archive[["w1", "w2", "w3", "w4"]].values.tolist()

strict_archive = archive[archive["final_structure"] == "strict_class"].copy()
syn_archive = archive[archive["final_structure"] == "synonym_cluster"].copy()
parts_archive = archive[archive["final_structure"] == "object_parts"].copy()
prop_archive = archive[archive["final_structure"] == "property_category"].copy()
phrase_archive = archive[archive["final_structure"] == "phrase_association"].copy()
domain_archive = archive[archive["final_structure"] == "domain_class"].copy()

STRICT_SEED_BANK = (
    strict_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

SYN_SEED_BANK = (
    syn_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

PARTS_SEED_BANK = (
    parts_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

PROP_SEED_BANK = (
    prop_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

PHRASE_SEED_BANK = (
    phrase_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

DOMAIN_SEED_BANK = (
    domain_archive.groupby("group")["words"]
    .apply(list)
    .to_dict()
)

In [ ]:
def safe_synset_lemmas(syn):
    lemmas = []
    for l in syn.lemmas():
        w = l.name().replace("_", " ").upper()
        if word_ok_synonym(w):
            lemmas.append(w)
    # unique, preserve order
    out = []
    seen = set()
    for w in lemmas:
        if w not in seen:
            seen.add(w)
            out.append(w)
    return out

def choose_diverse_words(words, k=4):
    chosen = []
    for w in words:
        if any(too_similar_surface(w, x) for x in chosen):
            continue
        chosen.append(w)
        if len(chosen) == k:
            return chosen
    return None

def good_synonym_label(words):
    # prefer more natural label
    head = min(words, key=lambda w: (polysemy_count(w), -token_zipf(w)))
    return f"SYNONYMS FOR {head}"

def synonym_candidates_strict(max_candidates=60):
    cands = []
    synsets = list(wn.all_synsets())
    random.shuffle(synsets)

    for syn in synsets:
        lemmas = safe_synset_lemmas(syn)
        if len(lemmas) < 4:
            continue

        # sort by frequency descending
        lemmas = sorted(lemmas, key=lambda w: (-token_zipf(w), len(w)))
        chosen = choose_diverse_words(lemmas, k=4)
        if not chosen:
            continue

        if group_has_surface_repetition(chosen):
            continue

        label = good_synonym_label(chosen)

        # reject ugly heads
        if label.endswith(" FOR THE") or label.endswith(" FOR A"):
            continue

        cands.append({
            "structure": "synonym_cluster",
            "group": label,
            "words": chosen,
            "source": "synonym_strict"
        })

        if len(cands) >= max_candidates:
            break

    return cands

In [ ]:
CURATED_STRICT_BANK = {
    "FRUITS": ["APPLE", "BANANA", "GRAPE", "MANGO", "PEACH", "PEAR", "PLUM"],
    "BIRDS": ["EAGLE", "HAWK", "RAVEN", "ROBIN", "SPARROW", "SWAN"],
    "TOOLS": ["DRILL", "HAMMER", "PLIERS", "SAW", "WRENCH", "VISE"],
    "FLOWERS": ["DAISY", "IRIS", "LILY", "ROSE", "TULIP", "VIOLET"],
    "BREAKFAST FOODS": ["BACON", "CEREAL", "EGGS", "PANCAKES", "TOAST", "WAFFLES"],
    "SOCIAL MEDIA APPS": ["DISCORD", "INSTAGRAM", "REDDIT", "SNAPCHAT", "TIKTOK", "X"],
    "BOARD GAMES": ["CHESS", "CLUE", "LIFE", "MONOPOLY", "RISK", "SORRY"],
    "WEB BROWSERS": ["CHROME", "EDGE", "FIREFOX", "OPERA", "SAFARI"],
    "PASTA SHAPES": ["ELBOW", "FUSILLI", "PENNE", "RIGATONI", "SHELL", "SPAGHETTI"],
}

def strict_candidates_from_bank(bank, source_name, max_candidates=80):
    cands = []
    items = list(bank.items())
    random.shuffle(items)

    for label, groups in items:
        # groups may be list of 4-word lists from archive, or a flat list in curated bank
        if groups and isinstance(groups[0], list):
            pool = []
            for g in groups:
                pool.extend(g)
        else:
            pool = list(groups)

        pool = [norm_word(w) for w in pool if word_ok_strict(w)]
        pool = list(dict.fromkeys(pool))
        pool = sorted(pool, key=lambda w: (-token_zipf(w), len(w)))

        chosen = choose_diverse_words(pool, 4)
        if not chosen:
            continue
        if group_has_surface_repetition(chosen):
            continue

        cands.append({
            "structure": "strict_class",
            "group": norm_label(label),
            "words": chosen,
            "source": source_name
        })

        if len(cands) >= max_candidates:
            break

    return cands

In [ ]:
def candidates_from_archive_bank(bank, structure_name, source_name, max_candidates=40, word_filter=word_ok_general):
    cands = []
    items = list(bank.items())
    random.shuffle(items)

    for label, groups in items:
        pool = []
        for g in groups:
            pool.extend(g)

        pool = [norm_word(w) for w in pool if word_filter(w)]
        pool = list(dict.fromkeys(pool))
        pool = sorted(pool, key=lambda w: (-token_zipf(w), len(w)))

        chosen = choose_diverse_words(pool, 4)
        if not chosen:
            continue
        if group_has_surface_repetition(chosen):
            continue

        cands.append({
            "structure": structure_name,
            "group": norm_label(label),
            "words": chosen,
            "source": source_name
        })

        if len(cands) >= max_candidates:
            break

    return cands

In [ ]:
def candidate_score(c, archive_group_counter=None):
    words = c["words"]
    label = c["group"]

    score = 0.0

    score += 1.2 * avg_token_zipf(words)
    score += 0.8 * label_quality(label)

    # reward compact familiar words
    score -= 0.08 * np.mean([len(w) for w in words])

    # repetition penalties
    if group_has_surface_repetition(words):
        score -= 5.0

    # punish weird synonym heads
    if c["structure"] == "synonym_cluster":
        if any(len(w.split()) > 2 for w in words):
            score -= 2.0
        if min(token_zipf(w) for w in words) < 3.9:
            score -= 2.0

    # small reward for archive-style labels
    if archive_group_counter is not None:
        score += 0.15 * archive_group_counter.get(label, 0)

    return score

In [ ]:
def canonical_words(words):
    return tuple(sorted(norm_word(w) for w in words))

def label_signature(label):
    toks = re.findall(r"[A-Z]+", norm_label(label))
    stop = {"OF", "FOR", "THE", "THAT", "ARE", "IN", "AT", "TO"}
    toks = [t for t in toks if t not in stop]
    return tuple(toks)

def dedupe_candidates_global(cands, max_per_structure=40):
    seen_wordsets = set()
    seen_labels = set()
    seen_label_sigs = set()
    per_structure = Counter()
    out = []

    for c in cands:
        wkey = canonical_words(c["words"])
        lkey = norm_label(c["group"])
        lsig = label_signature(c["group"])

        if wkey in seen_wordsets:
            continue
        if lkey in seen_labels:
            continue
        if lsig in seen_label_sigs and c["structure"] != "strict_class":
            continue
        if per_structure[c["structure"]] >= max_per_structure:
            continue

        seen_wordsets.add(wkey)
        seen_labels.add(lkey)
        seen_label_sigs.add(lsig)
        per_structure[c["structure"]] += 1
        out.append(c)

    return out

In [ ]:
archive_group_counter = Counter(archive["group"])

candidate_bank = []

# strict: archive first, curated second
candidate_bank += strict_candidates_from_bank(STRICT_SEED_BANK, "archive_strict", max_candidates=100)
candidate_bank += strict_candidates_from_bank(CURATED_STRICT_BANK, "curated_strict", max_candidates=60)

# property / parts / phrase / domain from archive
candidate_bank += candidates_from_archive_bank(PROP_SEED_BANK, "property_category", "archive_property", max_candidates=40)
candidate_bank += candidates_from_archive_bank(PARTS_SEED_BANK, "object_parts", "archive_parts", max_candidates=30)
candidate_bank += candidates_from_archive_bank(PHRASE_SEED_BANK, "phrase_association", "archive_phrase", max_candidates=30)
candidate_bank += candidates_from_archive_bank(DOMAIN_SEED_BANK, "domain_class", "archive_domain", max_candidates=30)

# synonyms: strict and capped
candidate_bank += synonym_candidates_strict(max_candidates=40)

for c in candidate_bank:
    c["score"] = candidate_score(c, archive_group_counter)

candidate_bank = sorted(candidate_bank, key=lambda x: x["score"], reverse=True)
candidate_bank = dedupe_candidates_global(candidate_bank, max_per_structure=35)

cand_df = pd.DataFrame(candidate_bank)
cand_df[["structure", "group", "words", "source", "score"]].head(40)

,structure,group,words,source,score
0,phrase_association,NBA PLAYERS,"[KING, SUN, MAGIC, THUNDER]",archive_phrase,7.682
1,property_category,SHADES OF BLUE,"[BABY, ROYAL, MIDNIGHT, POWDER]",archive_property,7.291
2,synonym_cluster,SYNONYMS FOR TOTALLY,"[ALL, WHOLE, COMPLETELY, TOTALLY]",synonym_strict,7.243
3,object_parts,BIKE PARTS,"[SPOKE, GEAR, BRAKE, SADDLE]",archive_parts,7.125
4,property_category,FAMOUS BROTHERS,"[WRIGHT, WARNER, JONAS, MARX]",archive_property,6.649
5,synonym_cluster,SYNONYMS FOR OUTCOME,"[EVENT, RESULT, EFFECT, OUTCOME]",synonym_strict,6.639
6,synonym_cluster,SYNONYMS FOR BUREAU,"[OFFICE, AGENCY, AUTHORITY, BUREAU]",synonym_strict,6.507
7,synonym_cluster,SYNONYMS FOR CREW,"[CREW, CROWD, BUNCH, GANG]",synonym_strict,6.381
8,property_category,OCCUPATIONAL SURNAMES,"[SMITH, MILLER, MASON, FISHER]",archive_property,6.375
9,synonym_cluster,SYNONYMS FOR GO ON,"[GO ON, MOVE ON, MARCH ON, PASS ON]",synonym_strict,6.367


In [ ]:
structure_probs = (
    archive["final_structure"]
    .value_counts(normalize=True)
    .to_dict()
)

def weighted_structure_sample(structure_probs, recent_structures=None):
    keys = list(structure_probs.keys())
    vals = np.array([structure_probs[k] for k in keys], dtype=float)

    if recent_structures:
        for i, k in enumerate(keys):
            if k in recent_structures[-2:]:
                vals[i] *= 0.65

    vals = vals / vals.sum()
    return np.random.choice(keys, p=vals)

def generate_green_batch_clean(cands, structure_probs, n=20, seed=7):
    random.seed(seed)
    np.random.seed(seed)

    by_structure = defaultdict(list)
    for c in cands:
        by_structure[c["structure"]].append(c)

    for s in by_structure:
        by_structure[s] = sorted(by_structure[s], key=lambda x: x["score"], reverse=True)

    chosen = []
    used_words = set()
    used_labels = set()
    recent_structures = []

    tries = 0
    while len(chosen) < n and tries < 5000:
        tries += 1
        s = weighted_structure_sample(structure_probs, recent_structures)

        if s not in by_structure or not by_structure[s]:
            continue

        bucket = by_structure[s][:min(20, len(by_structure[s]))]
        c = random.choice(bucket)

        wkey = canonical_words(c["words"])
        lkey = norm_label(c["group"])

        if wkey in used_words or lkey in used_labels:
            continue

        chosen.append(c)
        used_words.add(wkey)
        used_labels.add(lkey)
        recent_structures.append(s)

    return pd.DataFrame(chosen)

generated_green_df = generate_green_batch_clean(candidate_bank, structure_probs, n=45, seed=11)
generated_green_df[["structure", "group", "words", "source", "score"]]

,structure,group,words,source,score
0,property_category,FAMOUS BROTHERS,"[WRIGHT, WARNER, JONAS, MARX]",archive_property,6.649
1,synonym_cluster,SYNONYMS FOR GIVE WAY,"[GIVE WAY, FALL IN, FOUNDER, CAVE IN]",synonym_strict,5.670
2,synonym_cluster,SYNONYMS FOR PUT IN,"[PUT IN, STORE, LAY IN, SALT AWAY]",synonym_strict,5.856
3,property_category,OCCUPATIONAL SURNAMES,"[SMITH, MILLER, MASON, FISHER]",archive_property,6.375
4,synonym_cluster,SYNONYMS FOR TURN DOWN,"[TURN DOWN, PASS UP, REFUSE, REJECT]",synonym_strict,5.477
5,property_category,SHADES OF BLUE,"[BABY, ROYAL, MIDNIGHT, POWDER]",archive_property,7.291
6,synonym_cluster,SYNONYMS FOR AUTO,"[CAR, MACHINE, AUTO, AUTOMOBILE]",synonym_strict,6.351
7,object_parts,BIKE PARTS,"[SPOKE, GEAR, BRAKE, SADDLE]",archive_parts,7.125
8,synonym_cluster,SYNONYMS FOR AT REST,"[AT REST, AT PEACE, ASLEEP, DECEASED]",synonym_strict,5.391
9,synonym_cluster,SYNONYMS FOR OUTCOME,"[EVENT, RESULT, EFFECT, OUTCOME]",synonym_strict,6.639


In [ ]:
!pip -q install transformers torch

from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=0  # use -1 if no GPU
)

prompt = """
Generate a simple category for a word puzzle (like NYT Connections green).
Then give 4 words that belong to that category.

Format:
Category: <category>
Words: w1, w2, w3, w4

Keep words common and easy.

Example:
Category: Things found in a kitchen
Words: knife, spoon, pan, stove

Now generate a new one.
"""

out = generator(prompt, max_length=100)
print(out[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'image-to-image', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'question-answering', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'visual-question-answering', 'vqa', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection', 'translation_XX_to_YY']"

In [ ]:
generated_green_df.shape

(25, 5)

In [ ]:
!pip -q install transformers torch accelerate

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

generator = pipeline(
    "text-generation",   # <- important change
    model=model,
    tokenizer=tokenizer,
    device=0  # or -1 if no GPU
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausa

In [ ]:
def generate_text(prompt, max_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=0.7
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = """
List 20 common things that are yellow. Comma-separated.
"""

In [ ]:

print(generate_text(prompt))

food straw toys flower balls leafs hair blue.gases lemon peel tree peary plants plant leaves


In [ ]:
!pip -q install transformers torch sentencepiece accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/flan-t5-large"   # or flan-t5-large if Colab GPU can handle it

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

def generate_text(prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # keep deterministic for cleaner output
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
prompt = "Name 20 everyday objects. They should be red"
print(generate_text(prompt))

t-shirt
